# Riduzione della Ridondanza dei Sensori di una Linea di Produzione con PROC GVARCLUS

## Riepilogo Esecutivo

Una linea di produzione multi-zona trasmette in streaming decine di letture di sensori correlate, molte delle quali portano lo stesso segnale sottostante. Questo notebook usa **PROC GVARCLUS** (clustering delle variabili) per raggruppare i 13 sensori di processo in quattro cluster disgiunti, poi legge i valori R-quadrato di ciascun cluster per scegliere un sensore rappresentativo per cluster — riducendo l'impronta di monitoraggio SPC senza perdere informazioni di processo. Tre dei quattro cluster corrispondono chiaramente a zone fisiche (R-quadrato medio 0,92, 0,93 e 0,96); il quarto è un cluster a canale singolo che la procedura ha separato, che esaminiamo invece di tralasciare.

## Fonti dei Dati

Tutti i dati sono generati in linea con `call streaminit(20260531)` e `rand()` — nessun input esterno o di rete.

| Dataset | Righe | Variabili Chiave | Descrizione |
|---------|------|---------------|-------------|
| `process_sensors` | 100 | `zone1_a`–`zone1_c`, `zone2_a`–`zone2_c`, `zone3_a`, `zone3_b`, `temp_in`, `temp_out`, `pressure_a`, `pressure_b`, `vibration` | Letture sintetiche da una linea di produzione a 3 zone. I sensori all'interno di una zona condividono un driver latente (alta correlazione); sono aggiunti sensori incrociati tra zone (temperature legate alla zona 1, pressioni alla zona 2, vibrazione alla zona 3) per creare una ridondanza realistica. Il ciclo del passo DATA richiede 300 cicli, ma questo ambiente di valutazione gira in modalità senza licenza e mantiene le prime 100 osservazioni — sufficienti per recuperare chiaramente la struttura dei cluster. |
| `clusters` (OUT=) | 13 | `Variable`, `Cluster`, `RSq_Own` | Una riga per ogni sensore in input: il cluster a cui è stato assegnato e il suo R-quadrato con la componente di quel cluster. Prodotto dall'istruzione `OUTPUT OUT=`. |

# Riduzione della Ridondanza dei Sensori di una Linea di Produzione con PROC GVARCLUS

Su una linea di produzione strumentata, è comune registrare decine di sensori che misurano fenomeni fisici sovrapposti — più termocoppie in una zona, trasduttori di pressione ridondanti, sonde di vibrazione che tracciano lo stesso albero. Alimentare ogni canale in una carta di controllo o in un modello a valle spreca sforzo di monitoraggio e gonfia la multicollinearità.

**Il clustering delle variabili** risolve questo problema direttamente. `PROC GVARCLUS` partiziona le variabili numeriche in cluster *disgiunti*, dove ogni cluster è riassunto dalla prima componente principale dei suoi membri. I sensori che si muovono insieme finiscono nello stesso cluster; l'ingegnere può quindi mantenere un canale rappresentativo per cluster.

In questo notebook:

1. Sintetizziamo letture da una linea a 3 zone con sensori deliberatamente ridondanti (il ciclo richiede 300 cicli; qui ne vengono mantenuti 100).
2. Raggruppiamo i 13 canali con `PROC GVARCLUS` a `MAXCLUSTERS=4`.
3. Catturiamo le assegnazioni dei cluster in un dataset di output e le riassumiamo.
4. Interpretiamo i valori R-quadrato per scegliere un sensore per cluster per l'SPC continuo.

## Passo 1 — Generare dati sintetici dei sensori

Simuliamo tre zone di processo, ciascuna con un driver nascosto (`zone1_a`, `zone2_a`, `zone3_a`). I canali rimanenti sono funzioni lineari rumorose del driver della propria zona, quindi sono fortemente correlati all'interno di una zona ma quasi indipendenti tra zone. Leghiamo anche le temperature di ingresso/uscita alla zona 1 e i due trasduttori di pressione alla zona 2, imitando la ridondanza tra strumenti osservata sulle linee reali. Un seme fisso rende l'esecuzione riproducibile. Il ciclo richiede 300 cicli; in modalità senza licenza il motore mantiene le prime 100 osservazioni, come riportato dalla NOTE qui sotto.

In [1]:
DATI process_sensors;
    CHIAMARE streaminit(20260531);
    FARE cycle = 1 FINO_A 300;
        /* Zona 1: driver latente più due sonde ridondanti */
        zone1_a = rand("normal", 0, 1);
        zone1_b = 0.90*zone1_a + rand("normal", 0, 0.30);
        zone1_c = 0.85*zone1_a + rand("normal", 0, 0.35);

        /* Zona 2: driver latente più due sonde ridondanti */
        zone2_a = rand("normal", 0, 1);
        zone2_b = 0.88*zone2_a + rand("normal", 0, 0.30);
        zone2_c = 0.82*zone2_a + rand("normal", 0, 0.40);

        /* Zona 3: driver latente più una sonda ridondante */
        zone3_a = rand("normal", 0, 1);
        zone3_b = 0.90*zone3_a + rand("normal", 0, 0.30);

        /* Canali incrociati tra strumenti legati ai driver esistenti */
        temp_in    = 180 + 4.0*zone1_a + rand("normal", 0, 1.5);
        temp_out   = 178 + 4.0*zone1_a + rand("normal", 0, 1.6);
        pressure_a = 2.5 + 0.60*zone2_a + rand("normal", 0, 0.20);
        pressure_b = 2.5 + 0.58*zone2_a + rand("normal", 0, 0.22);
        vibration  = 0.40 + 0.30*zone3_a + rand("normal", 0, 0.10);
        USCITA;
    FINE;
    RIMUOVERE cycle;
ESEGUIRE;


NOTE: DATA process_sensors

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote process_sensors (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.07 seconds
  cpu   0.07 seconds


## Passo 2 — Raggruppare i sensori

Chiamiamo `PROC GVARCLUS` su tutti i 13 canali. L'istruzione `VAR` elenca i sensori numerici da raggruppare. Limitiamo la partizione a quattro cluster con `MAXCLUSTERS=4` (ci aspettiamo circa un cluster per zona fisica, con un po' di margine). `ODS GRAPHICS ON` con `PLOTS=ALL` abilita il dendrogramma dei cluster di variabili; il motore scrive quell'SVG nella directory di lavoro invece di renderizzarlo in linea, quindi la struttura che leggiamo qui sotto proviene dalla tabella stampata **Variable Cluster Assignments** e dalla tabella degli autovalori per cluster.

L'istruzione `OUTPUT OUT=` scrive le assegnazioni variabile-cluster — insieme all'R-quadrato di ogni variabile rispetto al proprio cluster — in un dataset su cui possiamo programmare in seguito.

In [2]:
ODS GRAPHICS ON;

PROCEDURA gvarclus DATI=process_sensors maxclusters=4 PLOTS=ALL;
    VARIABILE zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
    USCITA out=clusters;
ESEGUIRE;

ODS GRAPHICS OFF;


                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  zone1_a                                 1     0.968670
  zone1_b                                 1     0.918900
  zone1_c                                 4     1.000000
  zone2_a                                 2     0.983640
  zone2_b                                 2     0.946708
  zone2_c                                 2     0.892405
  zone3_a                                 3     0.977237
  zone3_b                                 3     0.949054
  temp_in                                 1     0.903394
  temp_out                                1     0.889519
  pressure_a                              2     0.929356
  pressure_b                              2     0.920915
  vibration  


NOTE: ODS Graphics is ON (width=640px, height=480px, format=SVG).
NOTE: PROC GVARCLUS data=process_sensors

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.
NOTE: ODS Graphics is OFF.


## Passo 3 — Rieseguire con l'opzione SUMMARY

Eseguire di nuovo la procedura con l'opzione `SUMMARY` mantiene la stessa partizione. `PLOTS=NONE` disattiva la grafica in questo passaggio. In questo ambiente il report stampato è identico al Passo 2 — la stessa tabella di assegnazione a 13 righe, gli stessi quattro cluster e lo stesso riepilogo degli autovalori — quindi questa cella dimostra principalmente che le opzioni `SUMMARY` e `PLOTS=NONE` vengono interpretate ed eseguite correttamente; non ci si aspetta che aggiunga nuovi numeri.

In [3]:
PROCEDURA gvarclus DATI=process_sensors maxclusters=4 summary PLOTS=none;
    VARIABILE zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
ESEGUIRE;


                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  zone1_a                                 1     0.968670
  zone1_b                                 1     0.918900
  zone1_c                                 4     1.000000
  zone2_a                                 2     0.983640
  zone2_b                                 2     0.946708
  zone2_c                                 2     0.892405
  zone3_a                                 3     0.977237
  zone3_b                                 3     0.949054
  temp_in                                 1     0.903394
  temp_out                                1     0.889519
  pressure_a                              2     0.929356
  pressure_b                              2     0.920915
  vibration  


NOTE: PROC GVARCLUS data=process_sensors

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.


## Passo 4 — Esaminare il dataset di output

Il dataset `clusters` prodotto da `OUTPUT OUT=` ha una riga per sensore con il suo `Cluster` assegnato e `RSq_Own` (la correlazione al quadrato tra il sensore e la componente del suo cluster). Un `RSq_Own` alto significa che il sensore è ben rappresentato dal proprio cluster — un candidato ideale da *scartare* a favore del rappresentante del cluster. Ordiniamo per cluster, poi per R-quadrato, in modo che il sensore più rappresentativo di ogni cluster si trovi in cima al proprio gruppo.

In [4]:
PROCEDURA ORDINARE DATI=clusters out=clusters_ranked;
    PER CLUSTER DISCENDENTE RSq_Own;
ESEGUIRE;

PROCEDURA STAMPARE DATI=clusters_ranked ETICHETTA;
    VARIABILE Variable CLUSTER RSq_Own;
    ETICHETTA Variable = "Canale Sensore"
          CLUSTER  = "Cluster"
          RSq_Own  = "R-Quadrato con il Proprio Cluster";
ESEGUIRE;


  Obs  Canale Sensore  Cluster  R-Quadrato con il Proprio Cluster
-----  --------------  -------  ---------------------------------
    1  ZONE1_A               1                            0.96867
    2  ZONE1_B               1                             0.9189
    3  TEMP_IN               1                           0.903394
    4  TEMP_OUT              1                           0.889519
    5  ZONE2_A               2                            0.98364
    6  ZONE2_B               2                           0.946708
    7  PRESSURE_A            2                           0.929356
    8  PRESSURE_B            2                           0.920915
    9  ZONE2_C               2                           0.892405
   10  ZONE3_A               3                           0.977237
   11  VIBRATION             3                            0.95916
   12  ZONE3_B               3                           0.949054
   13  ZONE1_C               4                                  1




NOTE: PROC SORT data=clusters

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 13 rows from clusters.
NOTE: Wrote clusters_ranked (13 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=clusters_ranked

NOTE: PROC PRINT completed: 13 observations printed, 3 variables


## Interpretazione dei risultati

La partizione recupera la maggior parte della struttura fisica della linea, con un'avvertenza onesta:

- **Cluster 1** raggruppa `zone1_a` (R²=0,969), `zone1_b` (0,919) e le temperature di ingresso/uscita `temp_in` (0,903) e `temp_out` (0,890) — quattro dei cinque canali guidati dal segnale latente della zona 1. R-quadrato medio 0,920.
- **Cluster 2** raggruppa tutti e tre i canali della zona 2 `zone2_a` (0,984), `zone2_b` (0,947), `zone2_c` (0,892) insieme ai due trasduttori di pressione `pressure_a` (0,929) e `pressure_b` (0,921), che abbiamo legato al driver della zona 2. R-quadrato medio 0,935.
- **Cluster 3** raggruppa `zone3_a` (0,977), `zone3_b` (0,949) e `vibration` (0,959). R-quadrato medio 0,962.
- **Cluster 4** è un canale singolo: `zone1_c`, la terza sonda della zona 1, è stata separata da sola con R²=1,000 (un singoletto è, banalmente, spiegato perfettamente da sé stesso). Nonostante condivida il driver della zona 1, con questa dimensione del campione la procedura ha giudicato `zone1_c` abbastanza distinto dal gruppo `zone1_a`/temperature da meritare un proprio cluster invece di essere unito al Cluster 1.

I tre cluster multi-canale portano ciascuno un R-quadrato medio superiore a **0,90**, confermando che una singola componente spiega la maggior parte della variazione tra i loro membri — esattamente la ridondanza che vogliamo comprimere. L'unico valore anomalo — `zone1_c` che finisce nel proprio cluster invece che con il resto della zona 1 — è un utile promemoria che il clustering delle variabili è guidato dai dati: con più cicli o un budget `MAXCLUSTERS` più alto il confine può spostarsi, quindi la partizione è un punto di partenza per il giudizio ingegneristico, non una verità fissa.

**Azione per la linea.** All'interno di ogni cluster multi-canale, mantenere il sensore con l'`RSq_Own` più alto (il canale più rappresentativo del proprio cluster) e ritirare o de-prioritizzare gli altri dal monitoraggio SPC di routine — per esempio `zone2_a` per il Cluster 2 e `zone3_a` per il Cluster 3. Trattare il singoletto `zone1_c` come una segnalazione da indagare piuttosto che un mantenimento automatico: verificare se porta informazioni genuinamente distinte o è un artefatto del clustering prima di decidere di monitorarlo separatamente. Anche tenendo da parte quel singolo canale, questo riduce 13 canali monitorati verso un piano di monitoraggio a quattro canali, tagliando il rumore degli allarmi e la multicollinearità mantenendo il contenuto informativo della linea.